<a href="https://colab.research.google.com/github/kph4br/Walmart-Midterm/blob/main/notebooks/2026-10-23%20-%20Midterm_Checkpoint%2010-27%20Updates%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [1]:
team_setup = {
    'team_name': 'Walmart Warriors',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Nate', 'github': 'NBushey26'},
        {'name': 'Claire', 'github': 'kph4br'},
        {'name': 'Marlise', 'github': 'MLuc123'}
    ],
    'repo_url': 'https://github.com/kph4br/Walmart-Midterm',                           # your main team repo: https://github.com/...
    'charter_url': 'https://github.com/kph4br/Walmart-Midterm/blob/main/CHARTER.md',                        # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'Claire',           # nice to have
    'meeting_time': 'Friday at noon',                       # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': ['1) store_id has 60 distinct values, but there are only 12 actual stores',
                            '2) unit_price is stored as text, and 3,029 rows start with $',
                            '3) There are about 45 SKUs but but only about 20 actual products',
                            '4) Category has 37 values for only 10 categories (mixed capitalization, etc)',
                            '5) timestamp mixes five formats and about 887 rows are null']              # nice to have, five specific problems
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  done - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [2]:
import subprocess, os

REPO_PATH = '/content/Walmart-Midterm'      # your local clone of the team repo

if os.path.exists(REPO_PATH):
    subprocess.run(['git', '-C', REPO_PATH, 'pull'])
else:
    subprocess.run(['git', 'clone', 'https://github.com/kph4br/Walmart-Midterm', REPO_PATH])

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')
print(result.stderr.strip())

bbc010d Rename charter.md to CHARTER.md



### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [5]:
# TODO
import pandas as pd
import numpy as np

def load_and_clean_transactions(filepath='walmart_transactions.csv'):
    df = pd.read_csv(filepath)
    initial_rows = len(df)

    df['store_id'] = df['store_id'].str.upper().str.strip()

    if df['unit_price'].dtype == object:
        df['unit_price'] = df['unit_price'].astype(str).str.replace('$', '', regex=False).str.strip()
        df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

    valid_qty_mask = df['quantity'] > 0
    returns_count = (~valid_qty_mask).sum()
    df = df[valid_qty_mask].copy()

    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    df = df.drop_duplicates()

    final_rows = len(df)
    print(f"Initial rows: {initial_rows}")
    print(f"Removed negative quantities/returns: {returns_count}")
    print(f"Final rows after cleaning: {final_rows}")

    return df

cleaned_df = load_and_clean_transactions()


EmptyDataError: No columns to parse from file

### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [6]:
# TODO: print or display your decision log
decision_log = [
    {"Decision": "Store ID Normalization", "Action": "Converted all `store_id` values to uppercase and stripped whitespaces.", "Rows Affected": "100% of rows standardized"},
    {"Decision": "Price Cleaning", "Action": "Stripped '$' characters and coerced non-numeric price entries to NaN.", "Rows Affected": "~15 rows coerced/cleaned"},
    {"Decision": "Negative Quantities", "Action": "Removed transactions with negative quantities (treated as customer returns/adjustments).", "Rows Affected": "130 rows removed"},
    {"Decision": "Timestamp Parsing", "Action": "Parsed mixed-format datetime strings into standard datetime objects.", "Rows Affected": "All valid timestamps converted"}
]

decision_df = pd.DataFrame(decision_log)
display(decision_df)

,Decision,Action,Rows Affected
0,Store ID Normalization,Converted all `store_id` values to uppercase a...,100% of rows standardized
1,Price Cleaning,Stripped '$' characters and coerced non-numeri...,~15 rows coerced/cleaned
2,Negative Quantities,Removed transactions with negative quantities ...,130 rows removed
3,Timestamp Parsing,Parsed mixed-format datetime strings into stan...,All valid timestamps converted


Our teammates did have disagreements about the cleaning decisions, but based on out charter we talked through each of our own logics behind the decisions and came to a consensus. For example, we disagreed about the importance of some little things like periods, commas, etc. We set a general guideline that everything should be as "clean" as possible.

### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [7]:
# TODO
df_catalog = pd.read_csv('product_catalog.csv')
print(f"Initial distinct SKUs in catalog: {df_catalog['sku'].nunique()}")
print(f"Distinct canonical product names: {df_catalog['product_name'].nunique()}")

poptart_mapping = df_catalog[df_catalog['product_name'].str.contains('Pop-Tart|pop-tarts', case=False, na=False)]
print("\nLargest Consolidated Group Example (Pop-Tarts):")
print(poptart_mapping[['sku', 'product_name', 'category', 'supplier']])

Initial distinct SKUs in catalog: 45
Distinct canonical product names: 38

Largest Consolidated Group Example (Pop-Tarts):
             sku          product_name         category           supplier
0          PT-12  pop-tarts strawberry  Breakfast Foods  Johnson & Johnson
1     SKU#459812   Strawberry Pop-Tart        Breakfast           Roku Inc
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast           Sara Lee
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast         Campbell's


### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [8]:
import requests
import time
import pandas as pd

def clean_store_id(store_id):
  store_id=str(store_id).upper().strip()
  store_id=store_id.replace('-', ''). replace('_','').replace(' ','')
  letters=store_id[:2]
  numbers=store_id[2:]
  return letters + '-' + numbers

def get_weather(url,params, max_tries=3):
  for attempt in range(1, max_tries+1):
    try:
      response = requests.get(url, params=params)
      response.raise_for_status()
      print("API call worked on attempt", attempt)
      return response.json()
    except Exception as e:
      print("Attempt", attempt, "failed:", e)
      if attempt< max_tries:
        wait_time=2 * attempt
        print("Waiting", wait_time, "seconds before trying again...")
        time.sleep(wait_time)
  print("All attempts failed.")
  return None


cleaned_df['store_id']=cleaned_df['store_id'].apply(clean_store_id)

df_stores=pd.read_csv('/content/Walmart-Midterm/data/store_locations.csv')
df_stores['store_id']=df_stores['store_id'].apply(clean_store_id)
df_stores=df_stores.drop_duplicates(subset='store_id')

df_stores.loc[df_stores['store_id'] == 'FL-619', 'latitude'] = 30.4383
df_stores.loc[df_stores['store_id'] == 'FL-619', 'longitude'] = -84.2807

print("Distinct stores_ids in sales data:", cleaned_df['store_id'].nunique())
print("Distinct stores_ids in store data:", df_stores['store_id'].nunique())


first_date = cleaned_df['timestamp'].min().date()
last_date = cleaned_df['timestamp'].max().date()
print("Sales date range:", first_date, "to", last_date)


url="https://archive-api.open-meteo.com/v1/archive"
all_weather=[]

for i in range(len(df_stores)):
  store = df_stores.iloc[i]

  params ={
    "latitude": store['latitude'],
    "longitude": store['longitude'],
    "start_date": str(first_date),
    "end_date": str(last_date),
    "hourly": "temperature_2m,precipitation,wind_speed_10m",
    "timezone": 'auto'
}

  print("Getting weather for", store['store_id'], store['city'])
  data = get_weather(url, params)

if data is not None:
  weather_df = pd.DataFrame({
      'time': data['hourly']['time'],
      'temperature_c': data['hourly']['temperature_2m'],
      'precipitation_mm': data['hourly']['precipitation'],
      'wind_speed_10kmh': data['hourly']['wind_speed_10m']
  })

  weather_df['time'] = pd.to_datetime(weather_df['time'])
  weather_df['store_id'] = store['store_id']
  weather_df['source'] = "Open-Meteo Historical Weather API"
  all_weather.append(weather_df)

else:
  print(" Skipping", store['store_id'], "because the call failed")
time.sleep(1)

if len(all_weather)>0:
  weather_df=pd.concat(all_weather, ignore_index=True)
  print()
  print("Total weather rows:", len(weather_df))
  print("Stores with weather data:", weather_df['store_id'].nunique())
  display(weather_df.head())
else:
  weather_df=pd.DataFrame()
  print("No weather data to display")



NameError: name 'cleaned_df' is not defined

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [9]:
import matplotlib.pyplot as plt

chart_df = cleaned_df.dropna(subset=['timestamp']).copy()
print("rows with a valid timestamp:", len(chart_df))
print("Rows dropped because timestamp was missing:", len(cleaned_df) - len(chart_df))

chart_df['date']=chart_df['timestamp'].dt.date
daily_orders = chart_df.groupby('date').size()

print(daily_orders.describe())

plt.figure(figsize=(12,5))
plt.plot(daily_orders.index,daily_orders.values)
plt.title("Daily Order Volume")
plt.xlabel("Date")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


NameError: name 'cleaned_df' is not defined

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [10]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': 25402,        # an integer
    'decisions_logged': 4,           # how many entries in your log
    'canonical_products': 38,         # distinct products after consolidation
    'weather_api_rows': 'Not Started',           # rows returned by your API call
    'notebook_runs_clean': 'Not Started',       # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

REQUIRED
  PASS - team_setup_complete = True
  PASS - charter_committed = True
PROGRESS (fill in what you have)
  done - rows_after_cleaning = 25402
  done - decisions_logged = 4
  done - canonical_products = 38
  done - weather_api_rows = Not Started
  done - notebook_runs_clean = Not Started

required items complete: True
progress items done: 5 of 5


True

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

The single most likely thing to go wrong in the next few weeks is probably technical issues with Colab. We have already had techn

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** _..._
We are running into problems with Colab saving our work. Specifically, the different parts we are working on are not showing up for everyone, making us have to redo parts of the project.

**Plan, with an owner and a date:** _..._
We all plan on really trying to make sure that Colab is saying that our work is saved and backing up what we are working on so that even if Colab does not save our work we will not have to redo it.


### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | answered | _Nate_ |
| Q2 SKU consolidation | answered | _Nate_ |
| Q3 weather correlation | answered | _Nate_ |
| Q4 store and geography | _not started_ | _Claire and Marlise_ |
| Q5 signal or artifact | _not started_ | _Claire and Marlise_ |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.